# R16c step 8 · The graders' "destroyed" setting, re-tuned for destroyed IoU (your PC, no GPU)

Built 22 Nov 2026 at Abrar's go (option 1). It trains nothing and runs no model: everything comes from files earlier
runs saved. Cells 1 and 2 are E7 step 2c's, unchanged (but for the output folder), so the scores are computed the same
way, and cell 2 first checks they reproduce the published ones.

**Why:** each grader (seeds 42, 43, 44) adds a bias to its four scores before choosing; its "destroyed" bias is
**−4.0** (`R16c-results/step5/calibration.json`, `step5b/`). R16c step 5 chose the biases to maximise the average grade
IoU of held-out true buildings, where intact buildings dominate, so a wrong "destroyed" cost more there than a missed
one. The project is judged on destroyed IoU of the map. L0b step 2 found the grader right on only 66 of 100 destroyed
buildings, against 92 of 100 intact.

**What varies:** a shift Δ added to every grader's destroyed bias: Δ = 0 (today's graders), 0.25, 0.5, … 4.0 (no
penalty left). The other three biases stay. The map is C4, the chosen chain: R11 on xBD photos (no grader, so Δ changes
nothing there), find-then-grade on the sharper photos. Scores are the mean of the three graders' maps, as published.

**The rule, fixed before the sweep:**
1. **Pick on validation** (the 605 clean pairs): the Δ with the highest C4 destroyed IoU, among the Δs whose C4 mIoU is
   at most 0.01 below Δ = 0's. Ties within 0.002 go to the smaller Δ (the smaller change).
2. **Adopt only if the test pairs confirm it, once** (the 2,007 clean test pairs): C4's destroyed IoU with the picked
   Δ beats Δ = 0 with the middle 95% of 10,000 redraws of the pairs above 0 (as E7 step 2b), and its mIoU drops by at
   most 0.01. If the pick is Δ = 0, today's setting stays and the test isn't run.

**Expectation, fixed before the sweep:**
- cell 2 reproduces every published score;
- the pick is between Δ = 0.5 and Δ = 2.0;
- C4's validation destroyed IoU rises by 0.005 to 0.02 (from 0.3501). Only about 85 of the 605 pairs are sharper
  photos, so the change is diluted;
- on the test pairs (Turkey's many sharper photos carry it), a gain of a similar size. Whether it clears noise is about
  even odds.

In [1]:
# ── Cell 1 (E7 step 2c's, unchanged but for OUT): Inputs and helpers (run from Essential_task_delegation/Champion_Task) ──
import io
import json
import re
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

OUT = Path("R16c-results/step8")
OUT.mkdir(parents=True, exist_ok=True)
LABELS = ["intact", "damaged", "destroyed", "none"]        # the grader's labels, in its output order
SEEDS = (42, 43, 44)
R11_COLS = ["r11_background_px", "r11_intact_px", "r11_damaged_px", "r11_destroyed_px"]
GT_COLS = ["gt_background_px", "gt_intact_px", "gt_damaged_px", "gt_destroyed_px"]
N_RESAMPLES, RNG_SEED = 10_000, 0                          # as E7 step 2b
TIE = 0.002
XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")


def event_of(name):
    """E1's rule: 'hurricane_florence_00000261_post_disaster.png' -> 'hurricane_florence'."""
    m = XBD_NAME.match(Path(name).stem)
    if m:
        return m.group(1)
    m = OTHER_NAME.match(Path(name).stem)
    return re.sub(r"\d+$", "", m.group(1) if m else Path(name).stem)


def source_of(name):
    return "xBD" if XBD_NAME.match(Path(name).stem) else "other"


# The graders' label biases, from their calibration files (R16c steps 5 and 5b)
BIAS = {}
for path in ("R16c-results/step5/calibration.json", "R16c-results/step5b/calibration.json"):
    for seed, chosen in json.loads(Path(path).read_text(encoding="utf-8")).items():
        BIAS[int(seed)] = np.array([chosen["bias"][l.capitalize() if l != "none" else "None"] for l in LABELS])
assert sorted(BIAS) == list(SEEDS), BIAS

# ── Validation: the 605 clean pairs ──
leaks = pd.read_csv("E5-results/step1/val_train_leaks_hash.csv")
LEAKED = set(leaks.loc[leaks["counted"].astype(str) == "True", "val_image"])
pc = pd.read_csv("E5-results/step2/per_image_counts.csv")
r11v = pc[(pc["model"] == "R11") & (pc["mode"] == "tta")].set_index("image")
CM_COLS = [f"t{t}_p{p}" for t in range(4) for p in range(4)]
VAL_IMAGES = sorted(set(r11v.index) - LEAKED)
R11_VAL = {i: r11v.loc[i, CM_COLS].to_numpy(np.int64).reshape(4, 4) for i in VAL_IMAGES}
GT_VAL = {i: cm.sum(1) for i, cm in R11_VAL.items()}         # true pixels per class, per image
iv5 = pd.read_csv("R16c-results/step5/instances_val.csv").query("found_by == 'sam3'")
iv5b = pd.read_csv("R16c-results/step5b/instances_val.csv").query("found_by == 'sam3'")
IV = iv5.merge(iv5b[["image", "id"] + [f"logp_{l}_s44" for l in LABELS]], on=["image", "id"], how="left", validate="one_to_one")
assert IV[[f"logp_{l}_s44" for l in LABELS]].notna().all().all(), "seed 44's grades don't cover every outline"
IV = IV[IV["image"].isin(VAL_IMAGES)].reset_index(drop=True)

# ── Test: the 2,007 clean pairs (E6), with R11's saved maps and the true masks ──
inputs = zipfile.ZipFile("E7-inputs/e7-inputs.zip")
pairs = pd.read_csv(io.BytesIO(inputs.read("test_pairs.csv")))
TEST_IMAGES = sorted(pairs.loc[pairs["clean"].astype(str) == "True", "after_image"])
masks = zipfile.ZipFile(io.BytesIO(inputs.read("test_masks_4class.zip")))
r11_maps = zipfile.ZipFile("E7-results/step2/r11_tta_maps.zip")
GT_TEST, R11_TEST = {}, {}
for name in TEST_IMAGES:
    pred = cv2.imdecode(np.frombuffer(r11_maps.read(name), np.uint8), cv2.IMREAD_GRAYSCALE)
    gt = cv2.imdecode(np.frombuffer(masks.read(name), np.uint8), cv2.IMREAD_GRAYSCALE)
    if gt.shape != pred.shape:                               # E7 read each mask at its image's size
        gt = cv2.resize(gt, (pred.shape[1], pred.shape[0]), interpolation=cv2.INTER_NEAREST)
    GT_TEST[name] = np.bincount(gt.ravel(), minlength=4)[:4]
    R11_TEST[name] = np.bincount(gt.ravel().astype(np.int64) * 4 + pred.ravel(), minlength=16).reshape(4, 4)
IT = pd.read_csv("E7-results/step2/instances_test.csv")
IT = IT[IT["image"].isin(TEST_IMAGES)].reset_index(drop=True)


# ── Helpers ──
def ftg_classes(df, seed):
    """Find-then-grade: each outline's class from the grader (with its biases): 1-3, or 0 for "not a building"."""
    label = (df[[f"logp_{l}_s{seed}" for l in LABELS]].to_numpy(np.float64) + BIAS[seed]).argmax(1)
    return np.where(label < 3, label + 1, 0)


def grader_probs(df, seed):
    """The grader's calibrated probabilities (with its biases), columns intact, damaged, destroyed, none."""
    z = df[[f"logp_{l}_s{seed}" for l in LABELS]].to_numpy(np.float64) + BIAS[seed]
    z -= z.max(1, keepdims=True)
    p = np.exp(z)
    return p / p.sum(1, keepdims=True)


def per_image_cms(df, cls, images, gt_totals):
    """Each image's 4x4 confusion (rows truth, columns prediction) for a map made of the outlines with these classes,
    background elsewhere: the true pixels under each outline go to its class, the rest of the image to background."""
    out = {i: np.zeros((4, 4), np.int64) for i in images}
    gt = df[GT_COLS].to_numpy(np.int64)
    for img, idx in df.groupby("image").indices.items():
        if img in out:
            for c in range(4):
                out[img][:, c] += gt[idx][cls[idx] == c].sum(0)
    for img in images:
        out[img][:, 0] += gt_totals[img] - out[img].sum(1)
    return out


def scores(cm):
    tp = np.diag(cm).astype(np.float64)
    iou = tp / np.maximum(cm.sum(0) + cm.sum(1) - tp, 1)
    b_tp = cm[1:, 1:].sum()
    return {"miou": iou.mean(), "destroyed_iou": iou[3], "building_iou": b_tp / max(b_tp + cm[0, 1:].sum() + cm[1:, 0].sum(), 1)}


def total(cms, images):
    return sum((cms[i] for i in images), np.zeros((4, 4), np.int64))


# Self-checks of the helpers on a made-up image: one outline of 10 true destroyed px graded destroyed, 5 true intact px outside
_df = pd.DataFrame({"image": ["a"], **{c: [v] for c, v in zip(GT_COLS, [0, 0, 0, 10])}})
_cm = per_image_cms(_df, np.array([3]), ["a"], {"a": np.array([100, 5, 0, 10])})["a"]
assert _cm[3, 3] == 10 and _cm[1, 0] == 5 and _cm[0, 0] == 100 and _cm.sum() == 115
assert abs(scores(_cm)["destroyed_iou"] - 1) < 1e-12 and source_of("turkey_earthquake0_post_380.png") == "other"
print(f"✓ Validation: {len(VAL_IMAGES)} clean pairs, {len(IV):,} SAM 3 outlines on {IV['image'].nunique()} of them")
print(f"✓ Test: {len(TEST_IMAGES):,} clean pairs, {len(IT):,} SAM 3 outlines on {IT['image'].nunique():,} of them; biases {dict((s, BIAS[s].tolist()) for s in SEEDS)}")

✓ Validation: 605 clean pairs, 26,175 SAM 3 outlines on 539 of them
✓ Test: 2,007 clean pairs, 137,760 SAM 3 outlines on 1,887 of them; biases {42: [0.0, -1.75, -4.0, -1.25], 43: [0.0, -1.0, -4.0, -2.0], 44: [0.0, -2.0, -4.0, -3.5]}


In [2]:
# ── Cell 2 (E7 step 2c's, unchanged): Check first: the rebuilt scores must reproduce the published ones (within 0.0005) ──
published = pd.read_csv("E7-results/step2/map_scores.csv")
pub = published[published["scope"] == "clean pairs"].set_index("method")
checks = []


def check(what, mine, theirs, tol=0.0005):
    checks.append({"check": what, "rebuilt": round(float(mine), 4), "published": round(float(theirs), 4),
                   "pass": abs(mine - theirs) <= tol})


# Seed 43 on validation is allowed 0.0015 (amended after the first run, 17 Nov, which stopped here: 0.3063 rebuilt against
# 0.3051 published). R16c step 5 recorded on 26 Oct that seed 43's per-building file rebuilds its maps only to within
# 0.0012: its saved log-probabilities are rounded, so near-ties can flip. Its published clean score used the lower value.
# Every other check stays at 0.0005. All candidates are rebuilt from the same files, so they compare like with like.
for seed, published_destroyed in zip(SEEDS, (0.3161, 0.3051, 0.2982)):          # R16c steps 5 and 5b, clean pairs
    s = scores(total(per_image_cms(IV, ftg_classes(IV, seed), VAL_IMAGES, GT_VAL), VAL_IMAGES))
    check(f"validation, find-then-grade seed {seed}: destroyed IoU", s["destroyed_iou"], published_destroyed,
          tol=0.0015 if seed == 43 else 0.0005)
s = scores(total(R11_VAL, VAL_IMAGES))
check("validation, R11 with TTA: destroyed IoU", s["destroyed_iou"], 0.2692)
check("validation, R11 with TTA: mIoU", s["miou"], 0.5805)
for seed in SEEDS:
    s = scores(total(per_image_cms(IT, ftg_classes(IT, seed), TEST_IMAGES, GT_TEST), TEST_IMAGES))
    check(f"test, find-then-grade seed {seed}: destroyed IoU", s["destroyed_iou"], pub.loc[f"chain_s{seed}", "destroyed_iou"])
    check(f"test, find-then-grade seed {seed}: mIoU", s["miou"], pub.loc[f"chain_s{seed}", "miou"])
s = scores(total(R11_TEST, TEST_IMAGES))
check("test, R11 with TTA: destroyed IoU", s["destroyed_iou"], pub.loc["R11_tta", "destroyed_iou"])
check("test, R11 with TTA: mIoU", s["miou"], pub.loc["R11_tta", "miou"])
CHECKS = pd.DataFrame(checks)
print(CHECKS.to_string(index=False))
if not CHECKS["pass"].all():
    raise RuntimeError("The rebuilt scores don't reproduce the published ones, so nothing below can be trusted. Stop.")
print("✓ Every rebuilt score reproduces its published value")

                                             check  rebuilt  published  pass
validation, find-then-grade seed 42: destroyed IoU   0.3161     0.3161  True
validation, find-then-grade seed 43: destroyed IoU   0.3063     0.3051  True
validation, find-then-grade seed 44: destroyed IoU   0.2982     0.2982  True
           validation, R11 with TTA: destroyed IoU   0.2692     0.2692  True
                    validation, R11 with TTA: mIoU   0.5805     0.5805  True
      test, find-then-grade seed 42: destroyed IoU   0.4055     0.4055  True
               test, find-then-grade seed 42: mIoU   0.5180     0.5179  True
      test, find-then-grade seed 43: destroyed IoU   0.3982     0.3983  True
               test, find-then-grade seed 43: mIoU   0.5121     0.5121  True
      test, find-then-grade seed 44: destroyed IoU   0.3657     0.3657  True
               test, find-then-grade seed 44: mIoU   0.5086     0.5087  True
                 test, R11 with TTA: destroyed IoU   0.3984     0.3984  True

In [3]:
# ── Cell 3: the sweep on the 605 clean validation pairs, and the rule's pick ──
DELTAS = [round(0.25 * k, 2) for k in range(17)]                  # 0, 0.25, ... 4.0
D_IDX = LABELS.index("destroyed")


def ftg_classes_d(df, seed, delta):
    """ftg_classes with delta added to this grader's destroyed bias."""
    shift = np.zeros(4)
    shift[D_IDX] = delta
    label = (df[[f"logp_{l}_s{seed}" for l in LABELS]].to_numpy(np.float64) + BIAS[seed] + shift).argmax(1)
    return np.where(label < 3, label + 1, 0)


def c4_cms(df, images, gt_totals, r11_cms, seed, delta):
    ftg = per_image_cms(df, ftg_classes_d(df, seed, delta), images, gt_totals)
    return {i: (r11_cms[i] if source_of(i) == "xBD" else ftg[i]) for i in images}


assert np.array_equal(ftg_classes_d(IV, 42, 0), ftg_classes(IV, 42))      # delta 0 is today's grader
SHARP_VAL = [i for i in VAL_IMAGES if source_of(i) == "other"]
rows = []
for d in DELTAS:
    per = [c4_cms(IV, VAL_IMAGES, GT_VAL, R11_VAL, s, d) for s in SEEDS]
    full = [scores(total(c, VAL_IMAGES)) for c in per]
    sharp = [scores(total(c, SHARP_VAL)) for c in per]
    rows.append({"delta": d, "c4_destroyed_iou": np.mean([x["destroyed_iou"] for x in full]),
                 "c4_miou": np.mean([x["miou"] for x in full]), "c4_building_iou": np.mean([x["building_iou"] for x in full]),
                 "sharper_photos_destroyed_iou": np.mean([x["destroyed_iou"] for x in sharp]),
                 "sharper_photos_miou": np.mean([x["miou"] for x in sharp])})
SWEEP = pd.DataFrame(rows)
SWEEP.round(4).to_csv(OUT / "sweep_validation.csv", index=False)
print(f"{len(VAL_IMAGES)} clean validation pairs, {len(SHARP_VAL)} of them sharper photos")
print(SWEEP.round(4).to_string(index=False))

base = SWEEP.iloc[0]
ok = SWEEP[SWEEP["c4_miou"] >= base["c4_miou"] - 0.01]
best = ok["c4_destroyed_iou"].max()
PICK = float(ok[ok["c4_destroyed_iou"] >= best - 0.002]["delta"].min())
print(f"\n✓ The rule picks delta = {PICK} (C4 destroyed IoU {SWEEP.set_index('delta').loc[PICK, 'c4_destroyed_iou']:.4f} "
      f"against {base['c4_destroyed_iou']:.4f} at delta 0)")

605 clean validation pairs, 85 of them sharper photos
 delta  c4_destroyed_iou  c4_miou  c4_building_iou  sharper_photos_destroyed_iou  sharper_photos_miou
  0.00            0.3501   0.5881           0.6822                        0.2628               0.4440
  0.25            0.3481   0.5876           0.6822                        0.2623               0.4435
  0.50            0.3467   0.5874           0.6822                        0.2628               0.4437
  0.75            0.3456   0.5873           0.6822                        0.2633               0.4436
  1.00            0.3410   0.5861           0.6822                        0.2585               0.4413
  1.25            0.3378   0.5851           0.6822                        0.2570               0.4391
  1.50            0.3326   0.5840           0.6822                        0.2528               0.4378
  1.75            0.3260   0.5819           0.6822                        0.2462               0.4336
  2.00            0.3159   0

In [4]:
# ── Cell 4: once on the 2,007 clean test pairs, with 10,000 redraws (as E7 step 2c) ──
TEST_ROWS, RES = pd.DataFrame(), pd.DataFrame()
if PICK == 0:
    print("The pick is delta 0: today's setting stays, and the test pairs aren't used.")
else:
    turkey = [i for i in TEST_IMAGES if event_of(i).startswith("turkey_earthquake")]
    sharp = [i for i in TEST_IMAGES if source_of(i) == "other"]
    per = {d: [c4_cms(IT, TEST_IMAGES, GT_TEST, R11_TEST, s, d) for s in SEEDS] for d in (0.0, PICK)}
    rows = []
    for d, cms_list in per.items():
        row = {"delta": d}
        for scope, images in (("all", TEST_IMAGES), ("sharper photos", sharp), ("Turkey", turkey)):
            s = [scores(total(c, images)) for c in cms_list]
            row[f"destroyed_iou ({scope})"] = float(np.mean([x["destroyed_iou"] for x in s]))
            row[f"miou ({scope})"] = float(np.mean([x["miou"] for x in s]))
        rows.append(row)
    TEST_ROWS = pd.DataFrame(rows)
    TEST_ROWS.round(4).to_csv(OUT / "test_confirmation.csv", index=False)
    print(TEST_ROWS.round(4).to_string(index=False))

    def parts(cms):
        tp = np.array([cms[i][3, 3] for i in TEST_IMAGES], np.float64)
        union = np.array([cms[i][3, :].sum() + cms[i][:, 3].sum() - cms[i][3, 3] for i in TEST_IMAGES], np.float64)
        return tp, union

    rng = np.random.default_rng(RNG_SEED)
    idx = rng.integers(0, len(TEST_IMAGES), size=(N_RESAMPLES, len(TEST_IMAGES)))
    counts = np.stack([np.bincount(r, minlength=len(TEST_IMAGES)) for r in idx])
    draws = {d: np.mean([(counts @ tp) / (counts @ un) for tp, un in map(parts, per[d])], axis=0) for d in per}
    diff = draws[PICK] - draws[0.0]
    t = TEST_ROWS.set_index("delta")
    RES = pd.DataFrame([{"comparison": f"delta {PICK} minus delta 0", "observed": round(float(t.loc[PICK, "destroyed_iou (all)"]
                         - t.loc[0.0, "destroyed_iou (all)"]), 4), "low_2.5%": round(float(np.percentile(diff, 2.5)), 4),
                         "high_97.5%": round(float(np.percentile(diff, 97.5)), 4), "share_above_0": round(float((diff > 0).mean()), 4)}])
    RES.to_csv(OUT / "resampling.csv", index=False)
    print("\n" + RES.to_string(index=False))

The pick is delta 0: today's setting stays, and the test pairs aren't used.


In [5]:
# ── Cell 5: the verdict and the README ──
def md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"


if PICK == 0:
    VERDICT = "today's setting stays: no shift of the destroyed bias beats it on validation"
else:
    t, r = TEST_ROWS.set_index("delta"), RES.iloc[0]
    confirmed = r["low_2.5%"] > 0 and t.loc[0.0, "miou (all)"] - t.loc[PICK, "miou (all)"] <= 0.01
    VERDICT = (f"adopt delta {PICK} (destroyed bias -4.0 -> {-4.0 + PICK}): the test pairs confirm it" if confirmed else
               f"today's setting stays: delta {PICK} won on validation but the test pairs don't confirm it beyond noise")
print("Verdict:", VERDICT)
lines = ["# R16c step 8: the graders' destroyed setting, re-tuned for destroyed IoU", "",
         "Run on your PC, no GPU, from saved files only. The question, the rule and the expectation are in "
         "[r16c-step8-destroyed-bias.ipynb](../../r16c-step8-destroyed-bias.ipynb)'s first cell, fixed before the sweep.", "",
         f"**Verdict:** {VERDICT}.", "",
         "## Check first: the rebuilt scores reproduce the published ones", "", md(CHECKS), "",
         "## The sweep on the 605 clean validation pairs (C4, mean of three graders)", "", md(SWEEP.round(4)), "",
         f"The rule's pick: delta = {PICK}.", ""]
if PICK:
    lines += ["## Once on the 2,007 clean test pairs", "", md(TEST_ROWS.round(4)), "",
              "## Is the change more than noise? (10,000 redraws of the pairs)", "", md(RES), ""]
lines += ["Files: `sweep_validation.csv`" + (", `test_confirmation.csv`, `resampling.csv`." if PICK else ".")]
(OUT / "README.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"✓ Wrote README.md to {OUT}")

Verdict: today's setting stays: no shift of the destroyed bias beats it on validation
✓ Wrote README.md to R16c-results\step8
